# 2 — Chat Completions

Now that we know the API works, let's actually talk to the LLM.

We'll cover:
1. A basic chat call
2. Understanding the response object
3. System messages — giving the model a persona
4. Temperature — controlling randomness
5. Other useful parameters

In [ ]:
from dotenv import load_dotenv
from mistralai import Mistral
import os

load_dotenv()

client = Mistral(api_key=os.getenv("mistral_api_key"))
model = os.getenv("mistral_llm_model")

import re
import builtins

_original_print = builtins.print

def strip_md(text: str) -> str:
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'\*(.+?)\*', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'#{1,6}\s*', '', text)
    text = re.sub(r'>\s?', '', text)
    text = re.sub(r'[-*+]\s', '', text)
    return text.strip()

def clean_print(*args, **kwargs):
    args = [strip_md(a) if isinstance(a, str) else a for a in args]
    _original_print(*args, **kwargs)

builtins.print = clean_print

## Basic call

The simplest possible chat completion — one user message in, one response out.

In [ ]:
response = client.chat.complete(
    model=model,
    messages=[
        {"role": "user", "content": "tell me a 1 sentence short story"}
    ],
)

print(response)

That's the raw response object — lots of metadata. The actual answer is buried in there. Let's extract it:

In [ ]:
print(response.choices[0].message.content)

We can also see how many tokens the call used:

In [ ]:
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")

## System message

The `system` role lets you set the model's behavior, persona, or instructions **before** the user speaks.

The model will follow these instructions for the entire conversation.

In [ ]:
response = client.chat.complete(
    model=model,
    messages=[
        {"role": "system", "content": "You are a pirate. Speak like a pirate in all responses."},
        {"role": "user", "content": "tell me a 1 sentence short story"}
    ],
)

print(response.choices[0].message.content)

Same question, completely different answer.

## Temperature

Temperature controls how random/creative the model's output is.

- **`0.0`** — deterministic, always picks the most likely token. Best for factual Q&A.
- **`0.7`** — balanced (the default). Good for general use.
- **`1.5+`** — very creative/chaotic. Fun, but unreliable for facts.

Let's ask the same question 3 times with a low temperature.

Note: we use a system message to force a short answer — otherwise the model loves to give us markdown lists, bold text, and a whole essay about fruit.

In [ ]:
for i in range(3):
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "user", "content": "write me a 1 sentence short story."},
        ],
        temperature=0.0,
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")

Now the same thing with high temperature:

In [ ]:
for i in range(3):
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "user", "content": "write me a 1 sentence short story."},
        ],
        temperature=1.5,
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")

## Other useful parameters

### `max_tokens` — limit response length

Useful to keep answers concise and control costs.

In [ ]:
response = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": "Explain how chocolate is made."}],
    max_tokens=50,
)

print(response.choices[0].message.content)
print(f"\n(finish reason: {response.choices[0].finish_reason})")

Notice the `finish_reason` is `length` — the model was cut off, it didn't finish naturally.

### `top_p` — nucleus sampling

An alternative to temperature. Instead of scaling all probabilities, `top_p` only considers the smallest set of tokens whose probabilities add up to `p`.

- `top_p=0.1` — only consider the top 10% most likely tokens (very focused)
- `top_p=1.0` — consider all tokens (default)

Generally, use **either** temperature **or** top_p, not both.

In [ ]:
for i in range(3):
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "user", "content": "write me a 1 sentence short story."},
        ],
        top_p=0.1,
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")

In [ ]:
for i in range(3):
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "user", "content": "write me a 1 sentence short story."},
        ],
        top_p=1,
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")